In [2]:
from datetime import datetime as dt
from datetime import timedelta
import requests
import json
import pandas as pd
import polars as pl
from bs4 import BeautifulSoup
from usp.tree import sitemap_tree_for_homepage
from IPython.display import display, HTML
import selenium
import ezregex as ez
from selenium import webdriver
import urllib
from selenium.webdriver.common.by import By
from pathlib import Path
import fitz  # PyMuPDF
import re
from docx import Document
import bs4
from pdfminer.high_level import extract_text
import pdfminer
import ddgs, asyncddgs
from ddgs import DDGS
import asyncio
import sqlite3
from urllib.parse import urlparse
import pickle
from datetime import datetime
from datetime import timedelta
from polars import col as c

%cd ../other_data/

/home/zeke/hello/SchoolData/other_data


In [2]:
# Get the authorizers noah found
noah_authorizers = pl.read_excel("Nationwide Charter School List.xlsx", sheet_id=3)

Could not determine dtype for column 4, falling back to string
Could not determine dtype for column 5, falling back to string
Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 7, falling back to string


In [3]:
DOCUMENT_TYPES = ['minutes', 'letter of intent', 'application', 'approval', 'charter renewal', 'modification application']

In [4]:
def query_body(restart_token=None):
  rtn = {
    "version": "1.0.0",
    "queries": [
      {
        "Query": {
          "Commands": [
            {
              "SemanticQueryDataShapeCommand": {
                "Query": {
                  "Version": 2,
                  "From": [
                    {
                      "Name": "s1",
                      "Entity": "SchoolsCharterOpen",
                      "Type": 0
                    }
                  ],
                  "Select": [
                    {
                      "Column": {
                        "Expression": {
                          "SourceRef": {
                            "Source": "s1"
                          }
                        },
                        "Property": "napcs_School_Name"
                      },
                      "Name": "SchoolsCharterOpen.napcs_School_Name",
                      "NativeReferenceName": "School Name"
                    },
                    {
                      "Column": {
                        "Expression": {
                          "SourceRef": {
                            "Source": "s1"
                          }
                        },
                        "Property": "ccd_Virtual_Detail"
                      },
                      "Name": "SchoolsCharterOpen.ccd_Virtual_Detail",
                      "NativeReferenceName": "Virtual"
                    },
                    {
                      "Column": {
                        "Expression": {
                          "SourceRef": {
                            "Source": "s1"
                          }
                        },
                        "Property": "City"
                      },
                      "Name": "SchoolsCharterOpen.City1",
                      "NativeReferenceName": "City1"
                    },
                    {
                      "Column": {
                        "Expression": {
                          "SourceRef": {
                            "Source": "s1"
                          }
                        },
                        "Property": "State"
                      },
                      "Name": "SchoolsCharterOpen.State1",
                      "NativeReferenceName": "State1"
                    },
                    {
                      "Column": {
                        "Expression": {
                          "SourceRef": {
                            "Source": "s1"
                          }
                        },
                        "Property": "Website"
                      },
                      "Name": "SchoolsCharterOpen.Website1",
                      "NativeReferenceName": "Website1"
                    }
                  ]
                },
                "Binding": {
                  "Primary": {
                    "Groupings": [
                      {
                        "Projections": [
                          0,
                          1,
                          2,
                          3,
                          4
                        ]
                      }
                    ]
                  },
                  "DataReduction": {
                    "DataVolume": 3,
                    "Primary": {
                      "Window": {
                        "Count": 500
                      }
                    }
                  },
                  "Version": 1
                },
                "ExecutionMetricsKind": 1
              }
            }
          ]
        },
        "CacheKey": "{\"Commands\":[{\"SemanticQueryDataShapeCommand\":{\"Query\":{\"Version\":2,\"From\":[{\"Name\":\"s1\",\"Entity\":\"SchoolsCharterOpen\",\"Type\":0}],\"Select\":[{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"s1\"}},\"Property\":\"napcs_School_Name\"},\"Name\":\"SchoolsCharterOpen.napcs_School_Name\",\"NativeReferenceName\":\"School Name\"},{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"s1\"}},\"Property\":\"ccd_Virtual_Detail\"},\"Name\":\"SchoolsCharterOpen.ccd_Virtual_Detail\",\"NativeReferenceName\":\"Virtual\"},{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"s1\"}},\"Property\":\"City\"},\"Name\":\"SchoolsCharterOpen.City1\",\"NativeReferenceName\":\"City1\"},{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"s1\"}},\"Property\":\"State\"},\"Name\":\"SchoolsCharterOpen.State1\",\"NativeReferenceName\":\"State1\"},{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"s1\"}},\"Property\":\"Website\"},\"Name\":\"SchoolsCharterOpen.Website1\",\"NativeReferenceName\":\"Website1\"}]},\"Binding\":{\"Primary\":{\"Groupings\":[{\"Projections\":[0,1,2,3,4]}]},\"DataReduction\":{\"DataVolume\":3,\"Primary\":{\"Window\":{\"Count\":500}}},\"Version\":1},\"ExecutionMetricsKind\":1}}]}",
        "QueryId": "",
      }
    ],
    "cancelQueries": [],
    "modelId": 4287083
  }
  if restart_token:
    rtn['queries'][0]['Query']['Commands'][0]['SemanticQueryDataShapeCommand']['Binding']['DataReduction']['Primary']['Window']['RestartTokens'] = restart_token
  return rtn

In [5]:
"""PowerBI JSON query API parsing

    These are my rough notes for pulling data from PowerBI JSON queries.
    I sketched this out after working with a query that joined two data sources
    and returned tabular data.

    There are two sections in the code below:

    1) A rough typing of the API response.
        It's not at all complete and probably has some mistakes.
        But since the response body is so large, it helped me to sketch this out
        to keep track of all the different objects I encountered.

    2) Code for parsing the response body.
        There are two main techniques used here, which I describe in the rest of the docs below.


    # Response parsing

    The main query results are contained in a dataset object in the response,
    with two caveats:

    1) The row values are often numeric, where the UI has a mix of numeric and string values.

    2) The row values have a variable number of columns from record to record.


    Methods for solving those described below, using the following example.

    ```json
    {
        "PH": [{
            "DM0": [
                {
                    "S": [
                        {
                            "N": "G0",
                            "T": 1,
                            "DN": "D0"
                        },
                        {
                            "N": "G1",
                            "T": 3
                        }
                    ],
                    "C": [0, 0.95]
                },
                {
                    "C": [1],
                    "R": 2
                }
            ]
        }],

        "ValueDicts": {
            "D0": ["Some value 1", "Some value 2", "Some value 3"]
        }

    }
    ```

    ## 1. Resolving display values

    Note that there is a `ValueDicts` object that contains lists of categorical values.
    Note also that the first record of the dataset contains an `S` field with a value that
    matches a key in `ValueDicts`.

    There may be multiple `ValueDicts`, and each column's `S` object may or may not reference one.

    The value substitution is straightforward.
    First, find the correct values list in `ValueDicts` based on the column's `S.DN` key.
    Then, if the row contains an integer value for the given column,
    substitute in the string value at the index given by that integer.

    In the example above, the value of `PH[0].DM0.S[0].DN` is `"D0"`.
    The second column descriptor (`PH[0].DM0.S[1]`) does not give a `DN` value.
    (Note also that `S[0].T` and `S[1].T` are different -- `T` presumably is a type,
    probably `string` and `float`, respectively.
    To complete the substitution, we would replace the `0` in the first record with `Some value 1`,
    and the `1` in the second record with `Some value 2`.

    The query I analyze only extracted up to 100 distinct values from each column into the `ValueDicts` lists.
    Any further values were embedded in record, so no substitution was required.

    ## 2. Unpacking variable-length rows

    The dataset contains `n` columns, which we know based on the length of `S` in the first record
    (since these are each column's metadata).

    Each record contains a list of values `C` with `len(C) <= n`.

    In the case that `C` has exactly `n` records,
    no processing (beyond value substitution as above) is needed.

    When `C` has fewer than `n` records, it usually has an `R` field assigned to it,
    and may also have a `Ø` field.
    Both `Ø` and `R` will look like some integer value.
    These are bitmasks, i.e. sequences of binary flags that instruct how to unpack the column.

    For example, if `R = 20`, the binary representation looks like `00010100`.
    As a bitmask, we will read this from right to left (which is more natural to write code for).
    So the first flag is off, the second is off, the third is **on**, and so forth.

    `R` tells us which columns in the record are recycled from the previous row.
    Specifically, if we are processing the value in row `y` and column `i`,
    then if the `i`th bit of `R` is `1`, we set `(y, i) = (y-1, i)`.

    `Ø` tells us which columns in the record are `null`.
    As above, if `Ø_i` is `1`, we set `(y, i) = null`.

    If both bits are unset, we set `(y, i)` to the next available value in the current record's `C` list.

    Lastly, even after unpacking `R`, we might have fewer than `n records.
    This seems to mean that the trailing values in the row are `null`
    (even though this is not necessarily indicated by `Ø`).
    So we will just pad with `None` until we have `n` records.

    In the example above, the second record has `R=2`.
    In binary, this looks like `10`.
    Reading the bitmask, the first flag is `0` -- so take it from the `C` list.
    The second flag is `1`, so we will recycle the second value of the preceding record, or `0.95`.
    So we have the unpacked row `[1, 0.95]`.
    After value substitution, this is `["Some value 2", 0.95]`.

    ## Future work

    There's lots more info in even a basic response, including type information.
    A more complete parser would consider this additional information and apply transformations accordingly.
    (For example to convert timestamps into a `datetime` type.)

    There are also much more complicated queries that PowerBI can run.
    I have no idea what subset of queries the techniques described here apply to!
"""

# unmodified from https://gist.github.com/jnu/ba98b5b42bb732d603969e7fb89778fe

######################################################
# Rough typing of the API response
#

from typing import TypedDict, Any, cast

class PowerBISDescriptor(TypedDict):
    """Describes a column in the results dataset.

    N is an ID for a name, T is probably a type, DN is a list of display values.
    """
    N: str # e.g. "G0"
    T: int # e.g. 1
    DN: str # e.g. "D0"

class PowerBISRow(TypedDict):
    """First row in the results set.

    C are values, S are column descriptors, Ø indicates nulls.
    """
    C: list[Any]
    S: list[PowerBISDescriptor]
    Ø: int | None

class PowerBICRow(TypedDict):
    """Row Values with R-compression.

    C are some values, R/Ø are masks used for compression.
    """
    C: list[Any]
    R: int | None
    Ø: int | None

PowerBIRow = PowerBISRow | PowerBICRow

class PowerBISelectColumnDescriptor(TypedDict):
    """Info about a column."""
    Kind: int
    Depth: int
    Value: str # e.g. "G0"
    Name: str
    GroupKeys: list[dict] # TODO

class PowerBISelectDescriptor(TypedDict):
    """Info about the query."""
    Select: list[PowerBISelectColumnDescriptor]
    Expressions: dict # TODO
    Version: int

class PowerBIDM(TypedDict):
    """Wrapper for resultset rows."""
    DM: list[PowerBIRow]

PowerBIValueDicts = dict[str, list[str]]
"""Lookup tables for string values in the dataset."""

class PowerBIDS(TypedDict):
    """A dataset."""
    N: str # e.g. "DS0"
    IC: bool
    HAD: bool
    PH: list[PowerBIDM]
    ValueDicts: PowerBIValueDicts

class PowerBIDSR(TypedDict):
    """Wrapper for a dataset."""
    Version: int
    MinorVersion: int
    DS: list[PowerBIDS]

class PowerBIData(TypedDict):
    """Wrapper for response data."""
    timestamp: str
    rootActivityId: str
    fromCache: bool
    metrics: dict
    descriptor: PowerBISelectDescriptor
    dsr: PowerBIDSR

ResultRow = dict[str, Any]
"""The target type for the transformation.

Keys are column headers, values are row values.
"""

######################################################
# Response parsing tools
#

def unpack_power_bi_rows(
        headers: dict[str, PowerBISelectColumnDescriptor],
        rows: list[PowerBIRow],
        value_dicts: PowerBIValueDicts | None = None,
        ) -> list[ResultRow]:
    """Inflate the rows from the Power BI API response.

    Applies the following transformations to the rows:
       - Inflating de-duplicated records via `R` bitmask
       - Value substitution via `value_dicts`
       - Zip column headers and values via `headers` descriptors.

    Args:
        headers - Descriptors from the query response indexed by ID
        rows - List of records from the query response
        value_dicts - Optional lists of categorical values indexed by ID

    Returns:
        List of records as {column: value} dicts
    """
    unpacked = list[ResultRow]()

    n = len(rows[0]['C'])
    prev_row = [None] * n

    col_names = [f"c{i}" for i in range(n)]
    col_values: list[list[str] | None] = [None] * n

    for row in rows:
        # Read column description from the `S` field (in the first row).
        if 'S' in row:
            for i in range(n):
                # Set column names from the headers
                print(f"row['S'][{i}]", row['S'][i])
                print("HEADERS", headers)
                print("HEADER", headers[row['S'][i]['N']])
                col_names[i] = headers[row['S'][i]['N']].get('Name', f"c{i}")
                # Set column distinct values if possible
                dn = row['S'][i].get('DN', None)
                if dn and value_dicts:
                    col_values[i] = value_dicts.get(dn, None)

        current_row = [None] * n
        copymask = row.get('R', 0)
        nullmask = row.get('Ø', 0)

        # `j` and `n_c` will track iteration through the C values.
        j = 0
        n_c = len(row['C'])
        colmask = 1
        # Combine the previous row with the current row according to the masks/lookups.
        for i in range(n):
            if copymask & colmask:
                # Copy the value from the previous row.
                current_row[i] = prev_row[i]
            elif nullmask & colmask:
                # Set the value to None.
                current_row[i] = None
            else:
                # They seem to just omit trailing nulls, so only bother
                # unpacking until we run out of C values.
                if j < n_c:
                    current_row[i] = raw_value = row['C'][j]
                    # Only the first 100 categorical values are extracted into the values list.
                    # The rest are embedded as raw values into the row.
                    if col_values[i] and raw_value is not None and isinstance(raw_value, int):
                        current_row[i] = col_values[i][raw_value]
                    j += 1
                # Update this column for future masks.
                # (Note we update even if the value is null.)
                prev_row[i] = current_row[i]

            # Shift to next column
            colmask <<= 1

        unpacked.append(dict(zip(col_names, current_row)))

    return unpacked

def parse_power_bi_response(response_json: dict) -> list[ResultRow]:
    """"Parse the full JSON body of a Query API response.

    Args:
        response_json: JSON returned from the PowerBI Query API

    Returns:
        List of dicts like {column: value}
    """
    data = cast(PowerBIData, response_json['results'][0]['result']['data'])
    headers = {col['Value']: col for col in data['descriptor']['Select']}
    # TODO(jnu): seems like they are set up to support multiple data sets
    ds = data['dsr']['DS'][0]
    value_dicts = ds.get('ValueDicts', None)
    # TODO(jnu): again suspicious of the PH list, only taking first element for now
    rows = ds['PH'][0]['DM0']
    return unpack_power_bi_rows(headers, rows, value_dicts=value_dicts)


# Example:
#
#     response = requests.post(url, headers={...}, json={...})
#     response.raise_for_status()
#     parse_power_bi_response(response.json())
#
# Note that the specific url, headers, and query JSON you need could be a whole 'nother problem.

In [6]:
# Get charter schools
url = 'https://wabi-us-east2-b-primary-api.analysis.windows.net/public/reports/querydata'
headers = {
    "X-PowerBI-ResourceKey": "e35bb00d-211b-4e97-ba30-1f57679aff71",
    "ActivityId": "23801149-e76c-4b31-a1ed-b83a0561d6bd"
}

def query(rt=None):
    resp = requests.post(url, json=query_body(rt), headers=headers)
    if resp.status_code != 200:
        raise Exception(resp)
    req_log.append(resp)
    return resp

if True:
    df = charter_schools = pl.read_csv('charter_schools.csv')
else:
    req_log = []
    rt = None
    data = []
    while True:
        try:
            d = query(rt).json()
        except Exception as e:
            print(e)
            break
        data.append(d)
        rt = d['results'][0]['result']['data']['dsr']['DS'][0]['RT']
    _data = []
    for d in data:
        _data += parse_power_bi_response(d)
    df = pl.DataFrame(_data).rename({
        'SchoolsCharterOpen.napcs_School_Name': 'School Name',
        'SchoolsCharterOpen.ccd_Virtual_Detail': 'Virtual',
        'SchoolsCharterOpen.City1': 'City',
        'SchoolsCharterOpen.State1': 'State',
        'SchoolsCharterOpen.Website1': 'Website'
    })
    df.write_csv('charter_schools.csv')
    charter_schools = df



In [7]:
# Get bonds from the other script for cross referencing
bonds = pl.read_csv('bonds.csv')

In [8]:
# Get Authorizers from PDF
if True:
    authorizers = pl.read_csv('2016_authorizers.csv')
else:
    rtn = []
    doc = fitz.open('2016_authorizers.pdf')
    header = None
    _authorizers = []
    for page_num, page in enumerate(doc):
        page: fitz.Page
        table = page.find_tables().tables[0].extract()
        header = table.pop(0)
        _authorizers += table

    authorizers = (pl.DataFrame(_authorizers, schema=header)
        .with_columns(name=pl.col('AUTHORIZER NAME').str.replace_all('\n', ' '))
        .drop('AUTHORIZER NAME')
    )

    authorizers.write_csv('2016_authorizers.csv')

In [9]:
# Search for authorizer websites
def search_for_authorizer_URL(authorizer_name, state):
    query = f'charter school "authorizer" {authorizer_name} official website'
    print(f'Searching for `{query}`...')
    return DDGS().text(query, max_results=10)

if True:
    # with open('auth_searches.pkl', 'rb') as f:
    #     auth_searches = pickle.load(f)
    auth_urls = pl.read_parquet('2016_searched_urls2.parquet')

else:
    auth_urls = []
    auth_searches = []
    for row in authorizers.iter_rows(named=True):
        search = search_for_authorizer_URL(row['name'], row['STATE'])
        auth_searches.append(search)
        new_row = row
        new_row['search'] = search
        auth_urls.append(new_row)

    with open('auth_searches.pkl', 'wb') as f:
        pickle.dump(auth_searches, f)

    names = []
    urls = []
    url = []
    for i in auth_urls:
        names.append(i['name'])
        urls.append([k['href'] for k in i['search']])
        url.append(i['search'][0]['href'])
    auth_urls = pl.DataFrame({
        'name': names,
        'urls': urls,
        'url': url,
    })
    auth_urls.write_parquet('2016_searched_urls2.parquet')



In [10]:
# Clean all the authorizers and combine them
if True:
    combined_authorizers = pl.read_parquet('combined_authorizers.parquet')
else:
    from_noah = (noah_authorizers
        .filter(pl.col('Minutes').is_not_null())
        .with_columns(
            source=pl.lit('noah'),
            # Because there shouldn't by any of those in a URL, it should make a list of 1
            found_urls=pl.col('Minutes').str.split(', ~~')
        )
        .rename({'Authorizer ': 'name', 'Minutes': 'url_guess'})
        .select('name', 'url_guess', 'source', 'found_urls')
    )
    from_pdf = (auth_urls
        # Replace url_guess with the first string in found_urls which isn't wikipedia
        .rename({'url': 'url_guess', 'urls': 'found_urls'})
        .with_columns(
            found_urls=pl.col('found_urls').list.eval(
                pl.when(pl.element().str.contains(
                    r'wikipedia|visitcalifornia|www.ca.gov|50states.com|ontheworldmap.com|britannica.com|www.usa.gov|\.pdf|qualitycharters.org|newschannel|yahoo.com|voiceofsandiego|tribune|bing.com|google.com'
                ))
                .then(pl.lit(None))
                .otherwise(pl.element())
            ).list.drop_nulls(),
        )
        .with_columns(
            url_guess=pl.col('found_urls').list.get(0, null_on_oob=True),
            source=pl.lit('nasca pdf'),
            state=pl.lit(None),
        )
        # Reorder
        .select('name', 'url_guess', 'source', 'found_urls')
    )
    combined_authorizers = pl.concat([from_pdf, from_noah])
    # combined_authorizers.write_excel('combined_authorizers.xlsx')
    combined_authorizers = (combined_authorizers
        .with_columns(confidence=
            # If noah found it, it's probably right
            pl.when(pl.col('source') == 'noah').then(1).otherwise(0.6)
            # If it's duplicated, that suspicious
            - pl.when(pl.col('url_guess').is_duplicated()).then(.1).otherwise(0)
            # If it's a .com, dock confidence
            - pl.when(pl.col('url_guess').str.contains('.com', literal=True)).then(.1).otherwise(0)
            # If it has "news" in it, dock confidence
            - pl.when(pl.col('url_guess').str.contains('news', literal=True)).then(.1).otherwise(0)
            # If it has "ballotpedia", dock confidence
            - pl.when(pl.col('url_guess').str.contains('ballotpedia', literal=True)).then(.2).otherwise(0)
            # If it has "linkedin", dock confidence
            - pl.when(pl.col('url_guess').str.contains('linkedin', literal=True)).then(.1).otherwise(0)
            - pl.when(pl.col('url_guess').str.contains('facebook', literal=True)).then(.1).otherwise(0)
        )
        # if url_guess is null, set confidence to 0
        .with_columns(
            confidence=pl.when(pl.col('url_guess').is_null()).then(0).otherwise(pl.col('confidence'))
        )
        # add state by joining with the authroizer
        .join(authorizers.select('name', 'STATE'), on='name', how='left')
        .rename({'STATE': 'state'})
    )
    combined_authorizers.write_parquet('combined_authorizers.parquet')
    combined_authorizers.write_excel('combined_authorizers.xlsx')

In [12]:
# Design & Fill the DBs -- running will wipe DBs
if False:
    # Add school data
    with sqlite3.connect('schools.db') as conn:
        conn.execute("DROP TABLE IF EXISTS schools")
        conn.execute("""
            CREATE TABLE IF NOT EXISTS schools (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT,
                state TEXT,
                virtual TEXT,
                website TEXT,
                city TEXT
            )
        """)
    (df
        .rename({'School Name': 'name', 'State': 'state', 'Virtual': 'virtual', 'Website': 'website', 'City': 'city'})
        .write_database('schools', 'sqlite:///schools.db', if_table_exists='append')
    )

    # Add authorizer data
    with sqlite3.connect('authorizers.db') as conn:
        conn.execute("DROP TABLE IF EXISTS authorizers")
        conn.execute("""
            CREATE TABLE IF NOT EXISTS authorizers (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT,
                state TEXT,
                source TEXT
            )
        """)

        # Add the urls table
        conn.execute("DROP TABLE IF EXISTS urls")
        conn.execute("""
            CREATE TABLE IF NOT EXISTS urls (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                authorizer_id INTEGER,
                base_url_guess TEXT,
                base_url_found TEXT,
                search_url TEXT,
                FOREIGN KEY (authorizer_id) REFERENCES authorizers(id)
            )
        """)

    # Split the urls into their own table
    auth_to_write = combined_authorizers.with_columns(
        found_urls=pl.col('found_urls').list.join(','),
        id=pl.int_range(1, pl.col('found_urls').len() + 1)
    )

    (auth_to_write
        .select('id', 'name', 'state', 'source')
        .write_database('authorizers', 'sqlite:///authorizers.db', if_table_exists='append')
    )


    (auth_to_write
        .select(authorizer_id='id', base_url_guess='url_guess', base_url_found='found_urls')
        .write_database('urls', 'sqlite:///authorizers.db', if_table_exists='append')
    )

    # Add some additional tables for scraping
    # Add an empty table for search url types
    with sqlite3.connect('authorizers.db') as con:
        con.executemany("""BEGIN;
            -- A table for all the search urls that we've found (using the in-situ search bar)
            DROP TABLE IF EXISTS search_url_types;
            CREATE TABLE IF NOT EXISTS search_url_types (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                auth_id INTEGER,
                url TEXT,
                type TEXT,
                tried BOOLEAN,
                note TEXT,
                FOREIGN KEY (auth_id) REFERENCES authorizers(id)
            );

            DROP TABLE IF EXISTS documents;
            CREATE TABLE IF NOT EXISTS documents (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                auth_id INTEGER,
                url TEXT,
                type TEXT,

                FOREIGN KEY (auth_id) REFERENCES authorizers(id)
            );
            COMMIT;
        """)



In [ ]:
# Look at the dbs
def summarize_db(db_path: str):
    print(f"\n=== Database: {db_path} ===")
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()

        # 1. Get all table names
        cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = [row[0] for row in cur.fetchall()]
        print(f"Tables: {tables}")

        for table in tables:
            if table == 'sqlite_sequence': continue
            print(f"\n=== Table: {table} ===")

            # 2. Show schema for the table
            # cur.execute(f"PRAGMA table_info({table})")
            # schema = cur.fetchall()
            # Print the dims
            # print(f"Dims: {len(cur.execute(f'SELECT * FROM {table}'))}")
            # print(f"Dims: {pl.read_database(f'SELECT * FROM {table} LIMIT 5', conn).shape}")

            # Display the head
            display(pl.read_database(f'SELECT * FROM {table}', conn, infer_schema_length=1000))


summarize_db('schools.db')
summarize_db('authorizers.db')


=== Database: schools.db ===
Tables: ['sqlite_sequence', 'schools']

=== Table: schools ===


id,name,state,virtual,website,city
i64,str,str,str,str,str
1,""" NORTHERN SUMMIT ACADEMY SHAST…","""CA""",null,null,"""Anderson"""
2,""" THE CROSSINGS""","""TN""",null,null,"""Antioch"""
3,""" THE LEARNING ACADEMY AT THE E…","""FL""","""Not Virtual""",null,"""JUPITER"""
4,""" VALLEY VIEW CHARTER MONTESSOR…","""CA""","""Not Virtual""",null,"""El Dorado Hills"""
5,"""100 ACADEMY OF EXCELLENCE - EL…","""NV""","""Not Virtual""",null,"""N Las Vegas"""
…,…,…,…,…,…
8173,"""ZENITH ACADEMY WEST""","""OH""","""Not Virtual""","""https://www.zenithacademy.org/…","""Columbus"""
8174,"""ZETA BRON MOUNT EDEN""","""NY""",null,"""https://zetaschools.org/""","""Bronx"""
8175,"""ZETA BRON TREMENT PARK ELEM""","""NY""","""Not Virtual""","""https://zetaschools.org/""","""Bronx"""



=== Database: authorizers.db ===
Tables: ['sqlite_sequence', 'authorizers', 'urls', 'search_url_types']

=== Table: authorizers ===


id,name,state,source
i64,str,str,str
1,"""Arkansas Charter Authorizing P…","""AR""","""nasca pdf"""
2,"""Arizona State Board for Charte…","""AZ""","""nasca pdf"""
3,"""Alameda City Unified School Di…","""CA""","""nasca pdf"""
4,"""Alameda County Office of Educa…","""CA""","""nasca pdf"""
5,"""Amador County Office of Educat…","""CA""","""nasca pdf"""
…,…,…,…
197,"""Nevada State Public Charter Sc…","""NV""","""noah"""
198,"""New Hampshire – State Board of…",null,"""noah"""
199,"""New Mexico""",null,"""noah"""



=== Table: urls ===


id,authorizer_id,base_url_guess,base_url_found,search_url
i64,i64,str,str,null
1,1,"""https://law.justia.com/codes/a…","""https://law.justia.com/codes/a…",null
2,2,"""https://asbcs.az.gov/about""","""https://asbcs.az.gov/about,htt…",null
3,3,"""https://www.clcschools.org/abo…","""https://www.clcschools.org/abo…",null
4,4,"""https://www.yumingschool.org/e…","""https://www.yumingschool.org/e…",null
5,5,"""https://www.nj.gov/education/c…","""https://www.nj.gov/education/c…",null
…,…,…,…,…
197,197,"""https://charterschools.nv.gov/…","""https://charterschools.nv.gov/…",null
198,198,"""https://www.education.nh.gov/w…","""https://www.education.nh.gov/w…",null
199,199,"""https://web.ped.nm.gov/bureaus…","""https://web.ped.nm.gov/bureaus…",null



=== Table: search_url_types ===


id,fk_id,url,type,is_authorizer
null,null,null,null,null


In [ ]:
class WebsiteSearcher:
    _options = Options()
    # _options.add_argument('--headless')
    _options.add_argument("--disable-gpu")
    driver = webdriver.Chrome(options=_options)
    _log = logging.getLogger('WebsiteSearcher')
    _handler = logging.StreamHandler()
    _handler.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
    _log.addHandler(_handler)
    _log.setLevel(logging.INFO)
    log = _log.info
    debug = _log.debug
    warn = _log.warning
    error = _log.error

    def __init__(self, which: Literal['authorizers', 'schools']):
        self.which = which
        self.conn = sqlite3.connect(f'{which}.db')

    def __del__(self):
        self.driver.quit()
        self.conn.close()

    def log_db(self, col, val):
        with self.conn.cursor() as cursor:
            cursor.execute("INSERT INTO scrape_log (?) VALUES (?)", (col, val))

    def _get_entity_id(self, name: str) -> int:
        with self.conn.cursor() as cursor:
            cursor.execute("SELECT id FROM ? WHERE name = ?", (self.which, name))
            row = cursor.fetchone()
            if not row:
                return
            return row[0]

    def _find_via_search(self, url: str, page: requests.Response=None) -> dict[str, str]:
        """ Use a search engine API to find pages """

    def _find_via_sitemap(self, url: str, name: str) -> dict[str, str]:
        """ Regex the sitemap to find pages """
        self.log(f'Finding pages via sitemap for {url}')
        url_base = urlparse(url).netloc

        entity_id = self._get_entity_id(name)

        def get_status():
            with self.conn.cursor() as cursor:
                cols = ('url', 'tried', 'note')
                cursor.execute("SELECT ? FROM search_url_types WHERE auth_id = ?", (cols, entity_id))
                return cursor.fetchall()

        def set_status(url: str=None, note: str='', tried: bool=True):
            with self.conn.cursor() as cursor:
                cursor.execute("INSERT INTO search_url_types (url, auth_id, type, tried, note) VALUES (?, ?, ?, ?, ?)", (url, school_id, doc_type, tried, note))

        status = get_status()
        if status:
            self.debug(f"Using {len(status)} cached sitemap urls")
            return status

        self.debug(f'Getting sitemap for {url_base}')
        tree = sitemap_tree_for_homepage(url_base)
        regex = ez.stringEndsWith('.pdf').compile()

        for page in tree.all_pages():
            # We will set tried to true later, when we choose to download it or not
            set_status(url=page.ulr, note='from sitemap', tried=False)
            if regex.match(page.url):
                self.debug(f'Found {page.url} in the sitemap, investigating...')

    # Untested
    def _find_via_in_situ_search_bar(self, url: str, name, doc_type: Literal[*doc_types]) -> list[str]:
        """ Use the search bar on the website to find pages """
        self.log(f'Finding pages via in-situ search bar for {url}')

        entity_id = self._get_entity_id(name)
        url_base = urlparse(url).netloc

        def get_status():
            with self.conn.cursor() as cursor:
                cols = ('url', 'tried', 'note')
                cursor.execute("SELECT ? FROM search_url_types WHERE auth_id = ? AND type = ?", (cols, entity_id, doc_type))
                row = cursor.fetchone()
                if row:
                    return dict(zip(cols, row))
                return {'url': None, 'tried': False, 'note': ''}

        def set_status(url: str=None, note: str='', tried: bool=True):
            with self.conn.cursor() as cursor:
                cursor.execute("INSERT INTO search_url_types (url, auth_id, type, tried, note) VALUES (?, ?, ?, ?, ?)", (url, entity_id, doc_type, tried, note))

        def get_search_page():
            self.log(f"Executing search on {url}")
            # if a search_url is already in the db, use it
            status = get_status()
            if status['url']:
                self.debug(f"Using cached url {status['url']}")
                return requests.get(status['url'])

            self.debug("Executing new search")
            self.driver.get(url)
            soup = bs4.BeautifulSoup(self.driver.page_source, 'html.parser')
            # See if we can find a nav bar first
            nav_bar = soup.find_all('nav')
            if nav_bar:
                # If there's multiple nav elements, get the last one (the most specific one)
                nav_bar = nav_bar[-1]
                self.debug(f"Found nav bar")
                # If we can, try to find something who's name, id, or class has *search* in it
                search_re = re.compile(r'*search*', re.IGNORECASE)
                search_bar = nav_bar.find('input', {'name': search_re, 'id': search_re, 'class': search_re})
                if search_bar: self.debug(f"Found search bar")
                else:
                    self.debug(f"No search bar found")
                    set_status(note="no search bar found")
                    return

                # Also try to find the associated (nearby) search button
                # search_button_re = re.compile(r'*search|submit*', re.IGNORECASE)
                # search_button = nav_bar.find('button', {'name': search_button_re, 'id': search_button_re, 'class': search_button_re})
                # Or, instead, just get the button that shares a parent
                search_button = search_bar.parent.parent.find('button')
                if search_button: self.debug(f"Found search button")
                else:
                    self.debug(f"No search button found")
                    set_status(note="no search button found")
                    return

                # if we found a search bar, try to use it
                if search_bar and search_button:
                    self.debug(f"Using search bar")
                    search_bar.click()
                    # TODO: review this at some point
                    search_bar.send_keys(doc_type)
                    search_button.click()
                    # Wait till the page is loaded (there's a body element in it)
                    time.sleep(.5)
                    self.driver.find_element(By.TAG_NAME, 'body')

                    set_status(url=self.driver.current_url, note='found')

                    return self.driver.page_source
                else:
                    self.debug(f"No nav bar found")
                    set_status(note="couldn't find nav bar")
                    return

        def get_search_results(results_page: str) -> list[str]:
            self.log(f"Loading search results for {results_url}")
            soup = bs4.BeautifulSoup(results_page, 'html.parser')
            rtn = []
            for link in soup.find_all('a', href=True):
                # Exclude any links in the footer or a nav bar
                l = link['href'].strip()
                self.debug(f"Found link {l}")
                if link.find_parents(['footer', 'nav']):
                    self.debug(f"Excluding link {l} because it's in a footer or nav bar")
                    continue
                if l == '/' or not l:
                    self.debug(f"Excluding link {l} because it's a root link")
                    continue
                # allow relative links, but not links to somewhere else
                if l.startswith(url_base) or l.startswith('/'):
                    self.debug(f"Adding link {l}")
                    rtn.append(l)
                else:
                    self.debug(f"Excluding link {l} because it's not on the same domain")
            return rtn

        def handle_results_link(link: str):
            self.log(f"Loading {link}")
            page = requests.get(link)
            if page.status_code == 200:
                self.log(f"Found {link}")
                self.log_db('url', link)

        search_results_page = get_search_page()
        if search_results_page:
            page_links = get_search_results(search_results_page)
            if page_links:
                for page_link in page_links:
                    handle_results_link(page_link)


    def _verify_page_to_name(self, url: str, name: str) -> bool:
        """ Attempt to verify that this is the correct homepage for the given school/authorizer """

    def search(self, websites:list) -> pl.DataFrame:
        """ Search the websites for pages corresponding to doc_type that have keywords in them """



In [14]:
df = combined_authorizers.filter(pl.col('source') == 'noah')
data = {}
failed = []
for row in df.iter_rows(named=True):
    if data.get(row['name'], False):
        print('Skipping', row['name'])
        continue


    url = urlparse(row['url_guess'])
    base_url = f'{url.scheme}://{url.netloc}'

    print('Processing', row['name'], 'with base_url', base_url)

    try:
        data[row['name']] = list(sitemap_tree_for_homepage(base_url + '/').all_pages())
    except:
        try:
            data[row['name']] = list(sitemap_tree_for_homepage(base_url).all_pages())
        except:
            print('Failed to get sitemap for', row['name'], 'with url', base_url)
            failed.append(row['name'])
            data[row['name']] = []


Processing Texas State Board of Education with base_url https://sboe.texas.gov


Parsing sitemap from URL https://sboe.texas.gov/sitemap failed: Unsupported root element 'html'.
Request for URL https://sboe.texas.gov/.sitemap.xml failed: 403 Forbidden
Request for URL http://www.alchartercommission.com/robots.txt failed: 404 Not Found


Processing Alabama Public Charter School Commission with base_url http://www.alchartercommission.com
Processing Arizona State Board for Charter Schools with base_url https://asbcs.az.gov


Request for URL https://asbcs.az.gov/admin/config/search/xmlsitemap failed: 403 Forbidden
Request for URL https://dese.ade.arkansas.gov/robots.txt failed: HTTPSConnectionPool(host='dese.ade.arkansas.gov', port=443): Max retries exceeded with url: /robots.txt (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))
Request for URL https://dese.ade.arkansas.gov/sitemap-index.xml.gz failed: HTTPSConnectionPool(host='dese.ade.arkansas.gov', port=443): Max retries exceeded with url: /sitemap-index.xml.gz (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))


Processing Arkansas Division of Elementary & Secondary Educaton with base_url https://dese.ade.arkansas.gov


...

Request for URL https://www.education.nh.gov/sitemap-index.xml.gz failed: 403 Forbidden
Request for URL https://www.education.nh.gov/sitemap failed: 403 Forbidden
Request for URL https://www.education.nh.gov/sitemap_index.xml failed: 403 Forbidden
Request for URL https://www.education.nh.gov/admin/config/search/xmlsitemap failed: 403 Forbidden
Request for URL https://www.education.nh.gov/.sitemap.xml failed: 403 Forbidden
Request for URL https://www.education.nh.gov/sitemap_news.xml.gz failed: 403 Forbidden
Request for URL https://www.education.nh.gov/sitemap/sitemap-index.xml failed: 403 Forbidden
Request for URL https://www.education.nh.gov/sitemap_news.xml failed: 403 Forbidden
Request for URL https://www.education.nh.gov/sitemap.xml.gz failed: 403 Forbidden
Request for URL https://www.education.nh.gov/sitemap-news.xml.gz failed: 403 Forbidden
Request for URL https://www.education.nh.gov/sitemap_index.xml.gz failed: 403 Forbidden
Request for URL https://www.education.nh.gov/sitemap.

Processing New Mexico with base_url https://web.ped.nm.gov
Processing New York State Board of Regents & SUNY Charter Schools Institute with base_url https://www.regents.nysed.gov


Request for URL https://www.regents.nysed.gov/admin/config/search/xmlsitemap failed: 403 Forbidden
Request for URL https://www.regents.nysed.gov/.sitemap.xml failed: 403 Forbidden


Processing North Carolina Charter Schools Advisory Board recommends, State Board of Education approves with base_url https://www.dpi.nc.gov


Request for URL https://www.dpi.nc.gov/admin/config/search/xmlsitemap failed: 403 Forbidden
Request for URL https://www.dpi.nc.gov/.sitemap.xml failed: 403 Forbidden


In [60]:
url_data = []
for k, v in data.items():
    if not v:
        url_data.append({"name": k, "url": df.filter(c('name') == k)['url_guess'].item(), "priority": None, 'last_modified': None, 'change_frequency': None})
    for url in v:
        url_data.append({"name": k, "url": url.url, "priority": url.priority, 'last_modified': url.last_modified, 'change_frequency': url.change_frequency})

In [65]:
pl.DataFrame(url_data)['name'].unique().to_list()

['New York State Board of Regents & SUNY Charter Schools Institute',
 'Massachusetts Board of Elementary and Secondary Education',
 'Montana',
 'Arizona State Board for Charter Schools',
 'Maryland',
 'Arkansas Division of Elementary & Secondary Educaton',
 'New Hampshire – State Board of Education',
 'Idaho Public Charter School Commission',
 'Georgia State Charter Schools Commission',
 'Mississippi',
 'Nevada State Public Charter School Authority',
 'Alabama Public Charter School Commission',
 'Maine',
 'New Mexico',
 'Texas State Board of Education',
 'Indiana Charter School Board',
 'Iowa Department of Education',
 'Missouri',
 'Hawaii State Public Charter School Commission',
 'Louisiana Board of Elementary and Secondary Education',
 'North Carolina Charter Schools Advisory Board recommends, State Board of Education approves',
 'Delaware Department of Education']

name,url_guess,source,found_urls,confidence,state
str,str,str,list[str],f64,str
"""Texas State Board of Education""","""https://sboe.texas.gov/state-b…","""noah""","[""https://sboe.texas.gov/state-board-of-education/sboe-meetings/meeting-agenda-current""]",1.0,null
"""Alabama Public Charter School …","""http://www.alchartercommission…","""noah""","[""http://www.alchartercommission.com/apcsc-document-archive/""]",0.9,null
"""Arizona State Board for Charte…","""https://asbcs.az.gov/meeting-d…","""noah""","[""https://asbcs.az.gov/meeting-dates-materials""]",1.0,"""AZ"""
"""Arkansas Division of Elementar…","""https://dese.ade.arkansas.gov/…","""noah""","[""https://dese.ade.arkansas.gov/Offices/office-of-school-choice-and-parent-empowerment/charter-schools/submitted-letters-of-intent-and-applications""]",1.0,null
"""Delaware Department of Educati…","""https://education.delaware.gov…","""noah""","[""https://education.delaware.gov/community/charter-school-support/approvals-modifications-renewals-closures/#approvals""]",1.0,"""DE"""
…,…,…,…,…,…
"""Nevada State Public Charter Sc…","""https://charterschools.nv.gov/…","""noah""","[""https://charterschools.nv.gov/News/2025_Charter_Application_Cycle/""]",1.0,"""NV"""
"""New Hampshire – State Board of…","""https://www.education.nh.gov/w…","""noah""","[""https://www.education.nh.gov/who-we-are/division-of-educator-and-analytic-resources/bureau-of-educational-opportunities/charter-schools/approved-public-charter-schools""]",1.0,null
"""New Mexico""","""https://web.ped.nm.gov/bureaus…","""noah""","[""https://web.ped.nm.gov/bureaus/public-education-commission/submitted-applications/""]",1.0,null


In [ ]:
{f"{k},{}" for k, v in data.items() if not v}

{'Alabama Public Charter School Commission',
 'Arkansas Division of Elementary & Secondary Educaton',
 'Hawaii State Public Charter School Commission',
 'Idaho Public Charter School Commission',
 'Indiana Charter School Board',
 'Louisiana Board of Elementary and Secondary Education',
 'Maine',
 'Maryland',
 'Massachusetts Board of Elementary and Secondary Education',
 'Mississippi',
 'Montana',
 'Nevada State Public Charter School Authority',
 'New Hampshire – State Board of Education'}

In [ ]:
(pl.DataFrame(url_data)
    .filter(
        (pl.col('last_modified').dt.date() > datetime.now() - timedelta(days=6*30)) |
        (pl.col('last_modified').is_null()))
    .select(['name', 'url'])
    .write_csv("authorizers.csv")
)

In [69]:
(pl.DataFrame(url_data)
    .filter(
        (pl.col('last_modified').dt.date() > datetime.now() - timedelta(days=6*30)) |
        (pl.col('last_modified').is_null()))
    .select(['name', 'url'])
).write_json("visited_urls.json")

In [7]:
import os

In [8]:
# Set env var RUST_BACKTRACE=1
os.environ['RUST_BACKTRACE'] = '1'

scraped = pl.read_json("visited_urls.json")

thread '<unnamed>' panicked at crates/polars-arrow/src/array/struct_/mod.rs:125:48:
called `Result::unwrap()` on an `Err` value: ComputeError(ErrString("The children must have an equal number of values.\n                         However, the values at index 7 have a length of 0, which is different from values at index 0, 60553."))


PanicException: called `Result::unwrap()` on an `Err` value: ComputeError(ErrString("The children must have an equal number of values.\n                         However, the values at index 7 have a length of 0, which is different from values at index 0, 60553."))

In [212]:
# Load the scraped data logs
with open("visited_urls.json", "r") as f:
    data = json.load(f)

# Modify them to match the schema a little better (make match_data a list instead of a dict)
for i in data:
    i['match_data'] = list(i['match_data'].values())

scraped = pl.from_dicts(data, schema={
    "url": pl.Utf8,
    "last_modified": pl.Utf8,
    "name": pl.Utf8,
    "last_visited": pl.Utf8,
    "type": pl.Utf8,
    "extension": pl.Utf8,
    "keyword_matches": pl.List(pl.Utf8),
    "match_data": pl.List(pl.Struct({
        "keyword": pl.Utf8,
        "text": pl.Utf8,
        "start": pl.Int64,
        "end": pl.Int64,
        "context": pl.Utf8,
        })),
    "downloads": pl.List(pl.Utf8),
    "is_document": pl.Boolean,
    "skipped": pl.Boolean,
    "status": pl.Utf8,
})

In [213]:
# Clean the scraped data
bad_domains = (
    # "google.com",
    # "googleusercontent.com",
    # "googleapis.com",
    # "google.com",
    # "googleusercontent.com",
    # "googleapis.com",
    # "cloud.google.com",
    # "policies.google.com",
    # "play.google.com",
    # "support.google.com",
    # "business.google.com",
    # "accounts.google.com",
    # "developers.google.com",
    "protobuf.dev",
    "github.com",
    "developers.googleblog.com",
    "air-esri.maps.arcgis.com",
    "developer.chrome.com",
    "societyawards.com",
    "www.youtube.com",
    "x.com",
    "stackoverflow.com",
    "womentechmakers.devpost.com",
    "tea.co1.qualtrics.com",
    "brand.youtube",
    "google.aip.dev",
    "support.github.com",
)

scraped = (scraped
    .with_columns(
        c('last_modified').str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S"),
        c('last_visited').str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S"),
        c('url').map_elements(lambda x: urlparse(x).netloc).alias('domain'),
        c('keyword_matches').list.unique()
    )
    .filter(~(c('domain').is_in(bad_domains) | c('domain').str.contains("google")))
)

sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.


In [ ]:
pl.Config.set_tbl_rows(4)
pl.Config.set_tbl_width_chars(50)

# scraped.filter(c('match_data').list.len() > 0)[0]['match_data'].to_list()

minutes_matches = scraped.filter(
    True
    & (c('match_data').list.len() > 0)
    & c('is_document')
    # & (c('extension') == 'pdf')
    & c('keyword_matches').list.contains("minutes")
    # & c('url').str.contains("minutes")
    & ~c('skipped')
)
minutes_matches

In [329]:
pl.Config.set_tbl_rows(4)
pl.Config.set_tbl_width_chars(50)

# scraped.filter(c('match_data').list.len() > 0)[0]['match_data'].to_list()

matches_of_interest = scraped.filter(
    True
    & (c('match_data').list.len() > 0)
    & c('is_document')
    # & (c('extension') == 'pdf')
    & ~c('skipped')
    & ~c('url').str.contains("faq")
    & ~c('url').str.contains("manual")
    & ~c('url').str.contains("responsibilities-and-consequences")
    & ~c('url').str.contains("disciplinary-report")
    & c('keyword_matches').list.set_intersection(
        ('refinance', 'merger', 'surrender', 'startup', 'new school', 'enrollment cap', 'letter of intent', 'charter proposal')
    ).list.len() > 0
    # & c('url').str.contains("minutes")
)
len(matches_of_interest)

22

In [356]:
# Generate the report
def print_match(data):
    return f"""Keyword: {data['text']}
    Context (characters {data['start']-300} - {data['end']+300}):
        ...{re.sub(r"(?:\n\n|(?:\s){3,})", "\n", data['context']).replace("\n", "\n        ")}...
    """
with open("report.txt", "w") as f:
    for i in matches_of_interest.iter_rows(named=True):
        print(f"""
Authorizer: {i['name']}
URL: {i['url']}
Document modified on {i['last_modified'].strftime("%m-%d-%Y") if i['last_modified'] else "Unknown"} (scraped on {i['last_visited'].strftime("%m-%d-%Y")})
keyword matches: {', '.join(set(i['keyword_matches']))}
Matches:
    {'\n    '.join(map(print_match, i['match_data']))}
        """, file=f)



boilerplate = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Report</title>
  <link
    href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css"
    rel="stylesheet"
    integrity="sha384-QWTKZyjpPEjISv5WaRU9OFeRpok6YctnYmDr5pNlyT2bRjXh0JMhjY6hW+ALEwIH"
    crossorigin="anonymous"
  >
  <style>
    details {{
        margin-bottom: 1rem;
        outline: 1px solid #ccc;
        border-radius: 0.25rem;
        padding: 0.5rem;
    }}
    summary {{
        cursor: pointer;
    }}
    /*Have nested details be indented*/
    details details {{
        margin-left: 1rem;
    }}
    </style>
</head>
<body class="container my-4">
  <h1>Texas State Authorizer Report</h1>
  <p>{text}</p>
</body>
</html>
"""

# Generate the report
def print_match_html(data):
    return f"""<p>Keyword: {data['text']}</p>
    <details>
    <summary>Context (characters {data['start']-300} - {data['end']+300}):</summary>
        ...{re.sub(r"(?:\n\n|(?:\s){3,})", "\n", data['context']).replace("\n", "\n        ")}...
    </details>
    """

s = ""
for i in matches_of_interest.iter_rows(named=True):
    s += f"""
    <details>
    <summary>
        <a href="{i['url']}">{i['url']}</a>
        <p>keyword matches: {', '.join(set(i['keyword_matches']))}</p>
    </summary>
    <p>Authorizer: {i['name'] or "Unknown"}</p>
    """
    if i['last_modified']:
        days = (datetime.now() - i['last_modified']).days
        s += f"<h6>Document is ~{days//30} months, {days%30} days old</h6>"
    s += f"""
    <details>
    <summary>Matches:</summary>
    {'\n    '.join(map(print_match_html, i['match_data']))}
    </details>
    </details>
        """
# <h6>Document modified on {i['last_modified'].strftime("%m-%d-%Y") if i['last_modified'] else "Unknown"} (scraped on {i['last_visited'].strftime("%m-%d-%Y")})</h6>
with open("report.html", "w") as f:
    f.write(boilerplate.format(text=s))

In [312]:
(scraped.filter('is_document')
    ['keyword_matches'].list.set_intersection(
    ('refinance', 'merger', 'surrender', 'startup', 'new school', 'enrollment cap', 'letter of intent', 'charter proposal')
).list.len() > 0).sum()

37

In [246]:
print(f"""
Total pages: {len(scraped)}
Total documents: {len(scraped.filter(c('is_document')))}
Pages found with keywords: {len(scraped.filter((c('keyword_matches').list.len() > 0)))}
Documents found with keywords: {len(scraped.filter(
    True
    & (c('match_data').list.len() > 0)
    & c('is_document')
    # & (c('extension') == 'pdf')
    # & c('keyword_matches').list.contains("minutes")
    # & c('url').str.contains("minutes")
    # & ~c('skipped')
))}
Documents found with the keyword "minutes": {len(scraped.filter(
    True
    & (c('keyword_matches').list.len() > 0)
    & c('is_document')
    # & (c('extension') == 'pdf')
    & c('keyword_matches').list.contains("minutes")
    # & c('url').str.contains("minutes")
    # & ~c('skipped')
))}
""")


Total pages: 17518
Total documents: 11676
Pages found with keywords: 4649
Documents found with keywords: 331
Documents found with the keyword "minutes": 74



In [301]:
pl.Config.set_tbl_rows(15)
doc_kw = (scraped
    # .filter(c('is_document'))
    .explode('keyword_matches')
    .group_by('keyword_matches', 'is_document')
    .count()
    .sort('count', descending=True)
    .fill_null('No Keywords')
)

/tmp/ipykernel_491710/643979907.py:6: DeprecationWarning:

`GroupBy.count` is deprecated. It has been renamed to `len`.



In [300]:
import plotly.express as px

px.bar(doc_kw.filter(
        (doc_kw['keyword_matches'] != 'No Keywords')
        & (doc_kw['is_document'] == True),
        # & (~((doc_kw['is_document'] == False) & (doc_kw['keyword_matches'].is_in(['approval', 'application', 'minutes', 'renewal'])))),
    ),
    x='keyword_matches',
    y='count',
    # color='is_document',
    title='Keyword Matches (just documents)',
    text_auto=True,
    labels={
        'keyword_matches': 'Keyword',
        'count': 'Count',
    },
    # template='plotly_dark',
    # width=1200,
    # height=800,
    # xaxis_title='Keyword',
    # yaxis_title='Count',
    # showlegend=False,
    barmode='group',
)#.update_layout(showlegend=False)


In [115]:
minutes_matches.write_parquet("minutes_matches.parquet")

In [ ]:
pl.Config.set_tbl_rows(50)   # show up to 50 rows
scraped['domain'].value_counts()

domain,count
str,u32
"""sboe.texas.gov""",1
"""tea.texas.gov""",1
"""helpdesk.tea.texas.gov""",1
"""www.nysed.gov""",1
"""tealprod.tea.state.tx.us""",1
"""rptsvr1.tea.texas.gov""",1
"""ritter.tea.state.tx.us""",1
"""www.texaseducationinfo.org""",1


In [63]:
scraped['is_document'].value_counts()

is_document,count
bool,u32
false,5842
true,11676


In [70]:
scraped

url,last_modified,name,last_visited,type,extension,keyword_matches,match_data,downloads,is_document,skipped,status,domain
str,datetime[μs],str,datetime[μs],str,str,list[str],struct[5],list[str],bool,bool,str,str
"""https://tea.texas.gov/finance-…",2024-01-30 23:28:00,null,2025-08-17 01:01:22,null,null,[],"{null,null,null,null,null}",[],true,true,"""skipped: old""","""tea.texas.gov"""
"""https://tea.texas.gov/finance-…",2024-01-30 23:27:59,null,2025-08-17 01:01:22,null,null,[],"{null,null,null,null,null}",[],true,true,"""skipped: old""","""tea.texas.gov"""
"""https://tea.texas.gov/finance-…",2024-01-30 23:23:58,null,2025-08-17 01:01:22,null,null,[],"{null,null,null,null,null}",[],true,true,"""skipped: old""","""tea.texas.gov"""
"""https://tea.texas.gov/finance-…",2024-01-30 23:24:02,null,2025-08-17 01:01:22,null,null,[],"{null,null,null,null,null}",[],true,true,"""skipped: old""","""tea.texas.gov"""
"""https://tea.texas.gov/finance-…",2024-01-30 23:28:01,null,2025-08-17 01:01:22,null,null,[],"{null,null,null,null,null}",[],true,true,"""skipped: old""","""tea.texas.gov"""
"""https://tea.texas.gov/finance-…",2024-01-30 23:24:01,null,2025-08-17 01:01:22,null,null,[],"{null,null,null,null,null}",[],true,true,"""skipped: old""","""tea.texas.gov"""
"""https://tea.texas.gov/finance-…",2024-01-30 23:23:59,null,2025-08-17 01:01:22,null,null,[],"{null,null,null,null,null}",[],true,true,"""skipped: old""","""tea.texas.gov"""
"""https://tea.texas.gov/finance-…",2024-01-30 23:23:58,null,2025-08-17 01:01:22,null,null,[],"{null,null,null,null,null}",[],true,true,"""skipped: old""","""tea.texas.gov"""
"""https://tea.texas.gov/finance-…",2024-01-30 23:23:56,null,2025-08-17 01:01:24,null,null,[],"{null,null,null,null,null}",[],true,true,"""skipped: old""","""tea.texas.gov"""
